In [1]:
import torch
import os
import sys
from datasets import load_dataset

sys.path.insert(0, "..")
from testbed.data import prepare_dataloader
import config

dataset = load_dataset(
    os.path.join(config.testbed_dir, "data", "vqav2"),
    split="validation",
    data_dir=".",
    images_dir=config.coco_dir,
    trust_remote_code=True,
)

hparams = {
    "batch_size": 1,
    "num_shots": 32,
    "dtype": torch.float16,
    "generate_args": {"num_beams": 3, "max_new_tokens": 5},
}

dataloader = prepare_dataloader(
    dataset,
    batch_size=hparams["batch_size"],
    num_shots=hparams["num_shots"],
    shuffle=True,
)

/home/jyc/ICLTestbed/dev/../testbed/data/common.py:61: UserWarning: /home/jyc/ICLTestbed/dev/v2_OpenEnded_mscoco_train2014_questions.json is not exists. The train split will not be loaded.
  warnings.warn(
/home/jyc/ICLTestbed/dev/../testbed/data/common.py:61: UserWarning: /home/jyc/ICLTestbed/dev/v2_OpenEnded_mscoco_test-dev2015_questions.json is not exists. The test-dev split will not be loaded.
  warnings.warn(
/home/jyc/ICLTestbed/dev/../testbed/data/common.py:61: UserWarning: /home/jyc/ICLTestbed/dev/v2_OpenEnded_mscoco_test2015_questions.json is not exists. The test split will not be loaded.
  warnings.warn(


Generating validation split: 0 examples [00:00, ? examples/s]

In [3]:
from transformers import BitsAndBytesConfig
from testbed.models import Idefics
from global_icv_encoder import GlobalICVEncoder

device = torch.device("cuda:3")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
lmm = Idefics(
    config.idefics_9b_path,
    model_args=dict(quantization_config=quantization_config),
    torch_dtype=torch.float16,
).to(device)
lmm.eval()
icv_encoder = GlobalICVEncoder(4096, 32)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
from testbed.data.coco import postprocess_generation
from testbed.data import prepare_caption_input
from tqdm import tqdm
import evaluate

total_cider = evaluate.load("Kamichanw/CIDEr")
result = []

for i, batch in tqdm(dataloader, desc=f"Evaluating {lmm.model_name} ..."):
    text, images = prepare_caption_input(batch)
    predictions = lmm.generate(text, images, **hparams["generate_args"])
    for pred, context in zip(predictions, batch):
        last_cap = context[-1]
        gt_captions = last_cap["sentences_raw"]
        prediction = postprocess_generation(pred)
        total_cider.add(prediction=prediction, reference=gt_captions)
        result.append(
            {
                "cocoid": last_cap["cocoid"],
                "raw_output": pred,
                "filename": last_cap["filename"],
                "sentences": last_cap["sentences_raw"],
                "prediction": prediction,
            }
        )

eval_result = total_cider.compute()
eval_result

In [5]:
from pathlib import Path
import exp_settings as setting
from pytorch_lightning.utilities import rank_zero_only
from pytorch_lightning.utilities.deepspeed import (
    convert_zero_checkpoint_to_fp32_state_dict,
)
import shutil


save_path = Path("../results/ckpt")
cpk_save_path = save_path / "last.pth"
output_file = save_path / "lightning_module.bin"
convert_zero_checkpoint_to_fp32_state_dict(cpk_save_path, output_file)


Processing zero checkpoint '../results/ckpt/last.pth/checkpoint'
Detected checkpoint of type zero stage ZeroStageEnum.gradients, world_size: 1
Parsing checkpoint created by deepspeed==0.15.1
Reconstructed Frozen fp32 state dict with 1015 params 4596963600 elements
Reconstructed fp32 state dict with 2 params 131104 elements
Saving fp32 state dict to ../results/ckpt/lightning_module.bin


In [10]:

checkpoint = torch.load(output_file)
sd = checkpoint["state_dict"]
sd = {n: pn for n, pn in sd.items() if not n.startswith("lmm")}
torch.save(sd, save_path / "icv_cpk.pth")
os.remove(output_file)
shutil.rmtree(
    cpk_save_path,
)

: 

In [9]:

sd

{'icv_encoder.icv': tensor([[-0.0097, -0.0122,  0.0123,  ..., -0.0039, -0.0022,  0.0013],
         [ 0.0068, -0.0137,  0.0381,  ..., -0.0179,  0.0075, -0.0024],
         [ 0.0154, -0.0019,  0.0114,  ..., -0.0160, -0.0048,  0.0068],
         ...,
         [ 0.0077, -0.0014, -0.0033,  ...,  0.0032,  0.0031, -0.0034],
         [ 0.0028,  0.0182, -0.0168,  ...,  0.0023,  0.0007, -0.0021],
         [ 0.0060, -0.0079, -0.0010,  ...,  0.0166, -0.0024, -0.0063]],
        requires_grad=True),
 'icv_encoder.alpha': tensor([0.1050, 0.1049, 0.1042, 0.1050, 0.1022, 0.1047, 0.1044, 0.1022, 0.1047,
         0.1039, 0.1029, 0.1050, 0.1050, 0.1037, 0.1050, 0.1037, 0.1050, 0.1044,
         0.1043, 0.1051, 0.1051, 0.1042, 0.1051, 0.1051, 0.1043, 0.1050, 0.1028,
         0.1045, 0.1051, 0.1050, 0.1051, 0.1040], requires_grad=True)}